In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install lpips

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.8/53.8 kB 2.9 MB/s eta 0:00:00


In [3]:
import os
from PIL import Image
import torchvision.transforms as transforms
from torch.utils.data import Dataset, DataLoader
import torch.nn as nn
import torch.nn.functional as F
import torch
import torch.optim as optim
from torchvision.models import vgg16
import lpips
import matplotlib.pyplot as plt
import time

In [4]:
train_transform = transforms.Compose([
        transforms.Resize(512),
        transforms.RandomCrop(512),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])

eval_transform = transforms.Compose([
        transforms.Resize(512),
        transforms.CenterCrop(512),
        transforms.ToTensor(),
        transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
        ])


class DTD_Dataset(Dataset):

    def __init__(self, root, file_list, transform=None, class_to_idx=None):
        self.root = root
        self.transform = transform

        with open(file_list, mode='r', encoding='utf-8') as f:
            self.files = [line.strip() for line in f if line.strip()]

        if class_to_idx is None:
          unique_classes = sorted({os.path.normpath(p).split(os.sep)[0] for p in self.files})
          self.class_to_idx = {class_name: i for i, class_name in enumerate(unique_classes)}
        else:
          self.class_to_idx = class_to_idx

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        relative_path = self.files[idx]
        image_path = os.path.join(self.root, relative_path)
        img = Image.open(image_path).convert('RGB')
        rel_norm = os.path.normpath(relative_path)
        string_label = rel_norm.split(os.sep, 1)[0]
        label = self.class_to_idx[string_label]

        if self.transform:
            img = self.transform(img)

        return img, label

In [5]:
path_images = "drive/MyDrive/DeepLearning/dtd/images"
path_labels = "drive/MyDrive/DeepLearning/dtd/labels"
train_dataset = DTD_Dataset(path_images, os.path.join(path_labels, "train1.txt"), train_transform)
class_to_idx = train_dataset.class_to_idx
val_dataset = DTD_Dataset(path_images, os.path.join(path_labels, "val1.txt"), eval_transform, class_to_idx=class_to_idx)
test_dataset = DTD_Dataset(path_images, os.path.join(path_labels, "test1.txt"), eval_transform, class_to_idx=class_to_idx)

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2)
test_loader = DataLoader(test_dataset, batch_size=8, shuffle=False, num_workers=2)

In [6]:
class Encoder(nn.Module):
    def __init__(self, hidden_dim=128, embedding_dim=256):
        super().__init__()
        self.conv1 = nn.Conv2d(3, hidden_dim // 2, kernel_size=4, stride=2, padding=1)              # 512x512 -> 256x256
        self.conv2 = nn.Conv2d(hidden_dim // 2, hidden_dim, kernel_size=4, stride=2, padding=1)     # 256x256 -> 128x128
        self.conv3 = nn.Conv2d(hidden_dim, hidden_dim * 2, kernel_size=4, stride=2, padding=1)      # 128x128 -> 64x64
        self.downsample = nn.Conv2d(hidden_dim * 2, embedding_dim, kernel_size=4, stride=4, padding=0)  # 64x64 -> 16x16

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.relu(self.conv3(x))
        x = self.downsample(x)
        return x

class Decoder(nn.Module):
    def __init__(self, embedding_dim=256, hidden_dim=128):
        super().__init__()
        self.upsample = nn.ConvTranspose2d(embedding_dim, hidden_dim * 2, kernel_size=4, stride=4, padding=0) #16x16 -> 64x64
        self.conv1 = nn.ConvTranspose2d(hidden_dim * 2, hidden_dim, kernel_size=4, stride=2, padding=1)       # 64x64 -> 128x128
        self.conv2 = nn.ConvTranspose2d(hidden_dim, hidden_dim // 2, kernel_size=4, stride=2, padding=1)      # 128x128 -> 256x256
        self.conv3 = nn.ConvTranspose2d(hidden_dim // 2, 3, kernel_size=4, stride=2, padding=1)               # 256x256 -> 512x512

    def forward(self, x):
        x = self.upsample(x)
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = self.conv3(x)
        x = torch.tanh(x)
        return x

class VectorQuantizer(nn.Module):
    def __init__(self, num_embeddings=1024, embedding_dim=256, commitment_cost=0.25):
        super().__init__()
        self.embedding_dim = embedding_dim
        self.num_embeddings = num_embeddings
        self.commitment_cost = commitment_cost

        self.embeddings = nn.Embedding(num_embeddings, embedding_dim)
        self.embeddings.weight.data.uniform_(-1/self.num_embeddings, 1/self.num_embeddings)

    def forward(self, z):
        z_flattened = z.permute(0, 2, 3, 1).contiguous()
        z_flattened = z_flattened.view(-1, self.embedding_dim)

        distances = (torch.sum(z_flattened**2, dim=1, keepdim=True)
                     + torch.sum(self.embeddings.weight**2, dim=1)
                     - 2 * torch.matmul(z_flattened, self.embeddings.weight.t()))

        encoding_indices = torch.argmin(distances, dim=1)
        encodings = F.one_hot(encoding_indices, self.num_embeddings).float()

        quantized = torch.matmul(encodings, self.embeddings.weight)
        quantized = quantized.view(z.shape[0], z.shape[2], z.shape[3], self.embedding_dim)
        quantized = quantized.permute(0, 3, 1, 2).contiguous()

        e_latent_loss = F.mse_loss(quantized.detach(), z)
        q_latent_loss = F.mse_loss(quantized, z.detach())
        loss = q_latent_loss + self.commitment_cost * e_latent_loss

        quantized = z + (quantized - z).detach()

        return quantized, loss


VQGAN:
- Generator: Encoder-VQ-Decoder. It takes a real texture image and tries to generate a flawless, high-resolution copy of it
- Discriminator: it looks at the original image and the generated copy, trying to figure out which one is the fake one

Perceptual Loss (Reconstruction Loss): LPIPS (Learned Perceptual Image Patch Similarity). It does not look at the raw pixel coords, but the real texture and the generated texture through a pre-trained net that knows how to recognize shapes, edges and textures. It compares the hidden feature layers (it looks at the same freq of lines, same style of roughness, etc.)

VGG16: we freeze weights because when Perceptual Loss is high and the gradient flows backwards, only the Generator is updates. It's a pretrained net, we are only using its specialized vision

$Loss_{VQGAN} = L_{recon} + \lambda \cdot L_{GAN} + L_{percep} + L_{codebook}$

where

- $L_{GAN}$ uses the Hinge Loss for both generator and discriminator, with adaptive weighting: $\lambda = \frac{\nabla _{G_{L}} [L_{recon} + L_{percep}]}{\nabla _{G_{L}} [L_{GAN} + ϵ]}$


In [7]:
class VQGAN(nn.Module):
    def __init__(self, hidden_dim=128, embedding_dim=256, num_embeddings=1024, commitment_cost=0.25):
        super().__init__()
        self.encoder = Encoder(hidden_dim, embedding_dim)
        self.vq = VectorQuantizer(num_embeddings, embedding_dim, commitment_cost)
        self.decoder = Decoder(embedding_dim, hidden_dim)

    def forward(self, x):
        z = self.encoder(x)
        quantized, vq_loss = self.vq(z)
        x_recon = self.decoder(quantized)
        return x_recon, vq_loss

In [8]:
class Discriminator(nn.Module):
    def __init__(self, hidden_dim=64):
        super().__init__()
        self.conv1 = nn.Conv2d(3, hidden_dim, kernel_size=4, stride=2, padding=1)                   # (3, 256, 256) -> (64, 128, 128)
        self.relu = nn.LeakyReLU(negative_slope=0.2, inplace=True)

        self.conv2 = nn.Conv2d(hidden_dim, hidden_dim * 2, kernel_size=4, stride=2, padding=1)      # (64, 128, 128) -> (128, 64, 64)
        self.norm1 = nn.BatchNorm2d(hidden_dim * 2)

        self.conv3 = nn.Conv2d(hidden_dim * 2, hidden_dim * 4, kernel_size=4, stride=2, padding=1)  # (128, 64, 64) -> (256, 32, 32)
        self.norm2 = nn.BatchNorm2d(hidden_dim * 4)

        self.conv4 = nn.Conv2d(hidden_dim * 4, 1, kernel_size=4, stride=1, padding=1)               # (256, 32, 32) -> (1, 31, 31)

    def forward(self, x):
        x = self.relu(self.conv1(x))
        x = self.relu(self.norm1(self.conv2(x)))
        x = self.relu(self.norm2(self.conv3(x)))
        x = self.conv4(x)
        return x

In [9]:
class PerceptualLoss(nn.Module):
    def __init__(self):
        super().__init__()
        VGG16 = vgg16(weights='DEFAULT').features
        self.slice = nn.Sequential(*list(VGG16.children())[:16]).eval()
        for param in self.slice.parameters():
            param.requires_grad = False

    def forward(self, real, fake):
        return F.mse_loss(self.slice(fake), self.slice(real))

def calculate_adaptive_weight(recon_loss, g_loss, last_layer_weights):
    recon_grads = torch.autograd.grad(recon_loss, last_layer_weights, retain_graph=True, allow_unused=True)[0]
    g_grads = torch.autograd.grad(g_loss, last_layer_weights, retain_graph=True, allow_unused=True)[0]

    if recon_grads is None:
      recon_norm = torch.tensor(0., device=last_layer_weights.device)
    else:
      recon_norm = torch.norm(recon_grads)

    if g_grads is None:
      g_norm = torch.tensor(0., device=last_layer_weights.device)
    else:
      g_norm = torch.norm(g_grads)

    lambda_weight = recon_norm / (g_norm + 1e-4)
    lambda_weight = torch.clamp(lambda_weight, 0.0, 1e4).detach()
    return lambda_weight

In [10]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generator = VQGAN().to(device)
discriminator = Discriminator().to(device)
optimizer_generator = optim.Adam(generator.parameters(), lr=2e-4, betas=(0.5, 0.9))
optimizer_discriminator = optim.Adam(discriminator.parameters(), lr=5e-5, betas=(0.5, 0.9))
#perceptual_loss_fn = lpips.LPIPS(net='vgg').to(device)
perceptual_loss_fn = PerceptualLoss().to(device)

Downloading: "https://download.pytorch.org/models/vgg16-397923af.pth" to /root/.cache/torch/hub/checkpoints/vgg16-397923af.pth


100%|██████████| 528M/528M [00:07<00:00, 77.1MB/s]


In [11]:
def set_requires_grad(model, requires_grad):
  for param in model.parameters():
    param.requires_grad = requires_grad

def train_step(generator, discriminator, perceptual_loss_fn, data, optimizer_g, optimizer_d, disc_start_step, current_global_step):
    # GENERATOR

    # disable discriminator gradients while training generator
    set_requires_grad(discriminator, False)

    optimizer_g.zero_grad()
    recon_batch, vq_loss = generator(data)
    recon_loss = F.mse_loss(recon_batch, data)
    percept_loss = perceptual_loss_fn(recon_batch, data)

    # dynamic adaptive weight scheduling
    if current_global_step >= disc_start_step:
        fake_outputs = discriminator(recon_batch)
        g_loss = -fake_outputs.mean()   # hinge loss

        last_layer_weights = generator.decoder.conv3.weight
        disc_weight = calculate_adaptive_weight(recon_loss + percept_loss, g_loss, last_layer_weights)

        total_g_loss = vq_loss + recon_loss + percept_loss + (disc_weight * g_loss)
    else:
        total_g_loss = vq_loss + recon_loss + percept_loss

    total_g_loss.backward()
    optimizer_g.step()
    set_requires_grad(discriminator, True)

    # DISCRIMINATOR
    total_d_loss = torch.tensor(0.0, device=data.device)
    if current_global_step >= disc_start_step:
        optimizer_d.zero_grad()

        real_outputs = discriminator(data)
        fake_outputs_d = discriminator(recon_batch.detach())

        loss_real = F.relu(1.0 - real_outputs).mean()
        loss_fake = F.relu(1.0 + fake_outputs_d).mean()

        total_d_loss = 0.5 * loss_real + 0.5 * loss_fake
        total_d_loss.backward()
        optimizer_d.step()

    return total_g_loss.item(), total_d_loss.item()


In [ ]:
def validate(generator, val_loader, device):
  generator.eval()
  recon_loss = 0.0
  percept_loss = 0.0

  for batch_idx, (data, _) in enumerate(val_loader):
    data = data.to(device)
    recon_batch, vq_loss = generator(data)
    recon_loss += F.mse_loss(recon_batch, data).item()
    percept_loss += perceptual_loss_fn(recon_batch, data).item()

  return recon_loss / len(val_loader), percept_loss / len(val_loader)


In [15]:
def visual_validation(generator, val_loader, device):
  generator.eval()
  encoding_indices = []
  unique_indices = []
  visual_samples = None

  num_embeddings = generator.vq.num_embeddings
  embedding_dim = generator.vq.embedding_dim
  embeddings = generator.vq.embeddings.weight

  with torch.no_grad():
    for batch_idx, (data, _) in enumerate(val_loader):
      data = data.to(device)
      recon_batch, vq_loss = generator(data)

      visual_samples = (data.cpu(), recon_batch.cpu())

      z = generator.encoder(data)
      z_flattened = z.permute(0, 2, 3, 1).contiguous().view(-1, embedding_dim)

      distances = (torch.sum(z_flattened**2, dim=1, keepdim=True)
                    + torch.sum(embeddings**2, dim=1)
                    - 2 * torch.matmul(z_flattened, embeddings.t()))

      encoding_indices = torch.argmin(distances, dim=1)

  # percentage of used vectors
  unique_indices = torch.unique(encoding_indices)
  util_percen = len(unique_indices) / num_embeddings * 100

  # perplexity
  counts = torch.bincount(encoding_indices, minlength=num_embeddings).float()
  probs = counts / counts.sum()
  perplexity = torch.exp(-torch.sum(probs * torch.log(probs + 1e-10)))

  print(f"Total Codebook Size:    {num_embeddings}")
  print(f"Unique Vectors Used:    {len(unique_indices)} / {num_embeddings}")
  print(f"Codebook Utilization:   {util_percen:.2f}%")
  print(f"Codebook Perplexity:    {perplexity.item():.2f} (Higher is better)")

  # visual reconstruction plotting
  real_imgs, recon_imgs = visual_samples
  num_displayed_imgs = min(4, real_imgs.shape[0])

  fig, axes = plt.subplots(2, num_displayed_imgs, figsize=(num_displayed_imgs * 3, 6))

  for i in range(num_displayed_imgs):
    real_img = real_imgs[i].permute(1, 2, 0).numpy()
    recon_img = recon_imgs[i].permute(1, 2, 0).numpy()

    real_plot = ((real_img + 1) / 2).clip(0, 1)
    recon_plot = ((recon_img + 1) / 2).clip(0, 1)

    axes[0, i].imshow(real_plot)
    axes[0, i].set_title(f"original {i+1}")
    axes[0, i].axis('off')

    axes[1, i].imshow(recon_plot)
    axes[1, i].set_title(f"reconstructed {i+1}")
    axes[1, i].axis('off')

  plt.show()

In [15]:
global_step = 0
disc_start_step = 1000
epochs = 20

save_path = '/content/drive/MyDrive/DeepLearning/dtd/checkpoints'

best_val_score = float('inf')

for epoch in range(epochs):
    generator.train()
    discriminator.train()
    epoch_start = time.time()

    total_loss_generator = 0.0
    total_loss_discriminator = 0.0

    # training
    for batch_idx, (data, _) in enumerate(train_loader):
        data = data.to(device)
        g_loss_val, d_loss_val = train_step(
            generator, discriminator, perceptual_loss_fn, data,
            optimizer_generator, optimizer_discriminator,
            disc_start_step, global_step)

        global_step += 1
        total_loss_generator += g_loss_val
        total_loss_discriminator += d_loss_val

    avg_g_loss = total_loss_generator / len(train_loader)
    avg_d_loss = total_loss_discriminator / len(train_loader)
    print(f"====> Epoch {epoch} Finished | Avg G-Loss: {avg_g_loss:.4f} | Avg D-Loss: {avg_d_loss:.4f}\n")

    epoch_save_path = os.path.join(save_path, f"epoch_{epoch}")

    # save generator model at each epoch
    gen_path = os.path.join(epoch_save_path, "generator")
    os.makedirs(gen_path, exist_ok=True)

    torch.save(generator.state_dict(), os.path.join(gen_path, "main_model.pth"))
    torch.save(optimizer_generator.state_dict(), os.path.join(gen_path, "optimizer.pth"))

    # save discriminator model at each epoch
    disc_path = os.path.join(epoch_save_path, "discriminator")
    os.makedirs(disc_path, exist_ok=True)

    torch.save(discriminator.state_dict(), os.path.join(disc_path, "main_model.pth"))
    torch.save(optimizer_discriminator.state_dict(), os.path.join(disc_path, "optimizer.pth"))

    # validation
    #recon_loss, percept_loss = validate(generator, val_loader, device)
    #print(f" validation MSE: {recon_loss:.6f} | validation LPIPS: {percept_loss}")

    #if recon_loss is not None:
    #  optimizer_generator.step(recon_loss)
    #  optimizer_discriminator.step(recon_loss)
    #if epoch % 5 == 0:
    #  visual_validation(generator, val_loader, device)

    #generator.eval()
    #with torch.no_grad():


====> Epoch 0 Finished | Avg G-Loss: 4.3693 | Avg D-Loss: 0.0000

====> Epoch 1 Finished | Avg G-Loss: 4.3226 | Avg D-Loss: 0.0000

====> Epoch 2 Finished | Avg G-Loss: 4.5012 | Avg D-Loss: 0.0000

====> Epoch 3 Finished | Avg G-Loss: 4.9608 | Avg D-Loss: 0.0000

====> Epoch 4 Finished | Avg G-Loss: 21.1154 | Avg D-Loss: 0.5975

====> Epoch 5 Finished | Avg G-Loss: 12.9077 | Avg D-Loss: 0.6748

====> Epoch 6 Finished | Avg G-Loss: 9.6039 | Avg D-Loss: 0.6406

====> Epoch 7 Finished | Avg G-Loss: 9.3123 | Avg D-Loss: 0.7391

====> Epoch 8 Finished | Avg G-Loss: 10.3644 | Avg D-Loss: 0.6268

====> Epoch 9 Finished | Avg G-Loss: 6.8849 | Avg D-Loss: 0.7350

====> Epoch 10 Finished | Avg G-Loss: 5.9764 | Avg D-Loss: 0.6167

====> Epoch 11 Finished | Avg G-Loss: 6.6743 | Avg D-Loss: 0.5946

====> Epoch 12 Finished | Avg G-Loss: 6.9186 | Avg D-Loss: 0.6652

====> Epoch 13 Finished | Avg G-Loss: 6.0298 | Avg D-Loss: 0.5549

====> Epoch 14 Finished | Avg G-Loss: 6.7318 | Avg D-Loss: 0.5726

==